# GPT Action 库：Sharepoint（返回文件用于数据分析/文档摘要）

## 简介

本页面为开发者构建特定应用的 GPT Action 提供说明和指南。在继续之前，请先确保熟悉以下信息：
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零开始构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

该解决方案使 GPT action 能够结合用户在 SharePoint 或 Office365 中可访问文件的上下文来回答用户的问题，它使用了 Microsoft Graph API 的[搜索功能](https://learn.microsoft.com/en-us/graph/api/resources/search-api-overview?view=graph-rest-1.0)和[检索文件](https://learn.microsoft.com/en-us/graph/api/driveitem-get?view=graph-rest-1.0\&tabs=http)的能力。它使用 Azure Functions 处理 Graph API 的响应，将其转换为人类可读的格式，或者整理成 ChatGPT 能理解的结构。这段代码仅作为方向性参考，你应根据自己的需求进行修改。

该解决方案利用了[在 Actions 中检索文件](https://platform.openai.com/docs/actions/sending-files)的能力，并像你直接将文件上传到对话中一样使用它们。Azure Function 返回一个 base64 字符串，ChatGPT 会将其转换为文件。该解决方案既能处理结构化数据，也能处理非结构化数据，但存在大小和数量方面的限制（参见[这里](https://platform.openai.com/docs/actions/sending-files)的文档）

### 价值 + 示例业务用例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力，直接连接到 Sharepoint 中的文件

**示例用例**：
- 用户需要查找哪些文件与某个主题相关
- 用户需要一个关键问题的答案，而答案深藏在文档之中

## 架构/示例

![](../../../images/solution_1.gif)


该解决方案使用一个 Node.js Azure Function，基于已登录的用户来：

1. 根据用户最初的问题，搜索用户有权访问的相关文件。

2. 对找到的每个文件，将其转换为 base64 字符串。

3. 按照 ChatGPT 所期望的结构（见[这里](https://platform.openai.com/docs/actions/sending-files/inline-option)）格式化数据。

4. 将其返回给 ChatGPT。随后 GPT 就可以像你把文件上传到对话中一样使用这些文件。

![](../../../images/solution_1_architecture.png)

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://www.microsoft.com/en-us/microsoft-365/sharepoint/collaboration
- 应用 API 文档：https://learn.microsoft.com/en-us/previous-versions/office/developer/sharepoint-rest-reference/

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 能够访问 Sharepoint 环境
- Postman（以及 API 和 OAuth 相关知识）


## 中间件信息

如果你按照[搜索文件概念指南](https://learn.microsoft.com/en-us/graph/search-concept-files)操作，[Microsoft Graph Search API](https://learn.microsoft.com/en-us/graph/search-concept-files) 会返回符合条件的文件的引用，而不是文件内容本身。因此需要使用中间件，而不是直接调用 MSFT 端点。

我们需要重新组织该 API 的响应，使其与[这里](https://platform.openai.com/docs/actions/getting-started/inline-option)所述的 `openaiFileResponse` 预期结构相匹配。

### 额外步骤

#### 设置 Azure Function

1. 按照 [Azure Function cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function) 中的步骤设置 Azure Function

#### 添加函数代码

现在你已经有了一个经过身份验证的 Azure Function，我们可以更新该函数以搜索 SharePoint / O365

2. 进入你的测试函数，粘贴[此文件](https://github.com/openai/openai-cookbook/blob/main/examples/chatgpt/sharepoint_azure_function/solution_one_file_retrieval.js)中的代码。保存函数。

> **这段代码仅作为方向性参考**——虽然它应该可以开箱即用，但其设计初衷是让你根据自己的需求进行定制（参见本文档末尾的示例）。

3. 前往左侧 **Settings** 下的 **Configuration** 选项卡，设置以下环境变量。注意，根据你的 Azure 界面，这些变量可能直接列在 **Environment Variables** 中。

    1. `TENANT_ID`：从上一节复制

    2. `CLIENT_ID`：从上一节复制

4. 前往 **Development Tools** 下的 **Console** 选项卡

    1. 在控制台中安装以下包

       1. `npm install @microsoft/microsoft-graph-client`

       2. `npm install axios`

5. 完成后，再次尝试从 Postman 调用该函数（POST 调用），在 body 中填入以下内容（使用你认为能产生结果的查询和搜索词）。

     ```json
    {
        "searchTerm": "<choose a search term>"
    }
    ```

6. 如果收到了响应，就可以用它来设置 Custom GPT 了！有关设置的更多详细信息，请参阅 Azure Function 页面的 ChatGPT 部分


## 更详细的演练


下面介绍该解决方案特有的设置说明和演练。完整代码可以在[这里](https://github.com/openai/openai-cookbook/blob/main/examples/chatgpt/sharepoint_azure_function/solution_one_file_retrieval.js)找到。

### 代码演练

下面逐一介绍该函数的各个部分。开始之前，请确保已安装所需的包并设置好环境变量（参见安装步骤部分）。


#### 实现身份验证

下面是几个将在函数中使用的辅助函数。


##### 初始化 Microsoft Graph 客户端

创建一个函数，使用访问令牌初始化 Graph 客户端。它将用于搜索 Office 365 和 SharePoint。

```javascript
const { Client } = require('@microsoft/microsoft-graph-client');

function initGraphClient(accessToken) {
    return Client.init({
        authProvider: (done) => {
            done(null, accessToken);
        }
    });
}
```

##### 获取 On-Behalf-Of（OBO）令牌

该函数使用现有的 bearer 令牌，向 Microsoft 身份平台请求 OBO 令牌。这样可以传递凭据，确保搜索只返回已登录用户可访问的文件。

```javascript
const axios = require('axios');
const qs = require('querystring');

async function getOboToken(userAccessToken) {
    const { TENANT_ID, CLIENT_ID, MICROSOFT_PROVIDER_AUTHENTICATION_SECRET } = process.env;
    const params = {
        client_id: CLIENT_ID,
        client_secret: MICROSOFT_PROVIDER_AUTHENTICATION_SECRET,
        grant_type: 'urn:ietf:params:oauth:grant-type:jwt-bearer',
        assertion: userAccessToken,
        requested_token_use: 'on_behalf_of',
        scope: 'https://graph.microsoft.com/.default'
    };

    const url = `https\://login.microsoftonline.com/${TENANT_ID}/oauth2/v2.0/token`;
    try {
        const response = await axios.post(url, qs.stringify(params), {
            headers: { 'Content-Type': 'application/x-www-form-urlencoded' }
        });
        return response.data.access\_token;
    } catch (error) {
        console.error('Error obtaining OBO token:', error.response?.data || error.message);
        throw error;
    }
}
```

#### 从 O365 / SharePoint 项目中检索内容

该函数获取驱动器项目的内容，将其转换为 base64 字符串，并重新组织以匹配 `openaiFileResponse` 格式。
```javascript
const getDriveItemContent = async (client, driveId, itemId, name) => {
   try
       const filePath = `/drives/${driveId}/items/${itemId}`;
       const downloadPath = filePath + `/content`
       // this is where we get the contents and convert to base64
       const fileStream = await client.api(downloadPath).getStream();
       let chunks = [];
           for await (let chunk of fileStream) {
               chunks.push(chunk);
           }
       const base64String = Buffer.concat(chunks).toString('base64');
       // this is where we get the other metadata to include in response
       const file = await client.api(filePath).get();
       const mime_type = file.file.mimeType;
       const name = file.name;
       return {"name":name, "mime_type":mime_type, "content":base64String}
   } catch (error) {
       console.error('Error fetching drive content:', error);
       throw new Error(`Failed to fetch content for ${name}: ${error.message}`);
   }
```

#### 创建处理请求的 Azure Function

有了这些辅助函数之后，Azure Function 将编排整个流程：对用户进行身份验证、执行搜索，并遍历搜索结果以提取文本，将文本的相关部分检索给 GPT。

**处理 HTTP 请求：** 函数首先从 HTTP 请求中提取 query 和 searchTerm。它会检查是否存在 Authorization 请求头，并提取 bearer 令牌。

**身份验证：** 使用 bearer 令牌，通过上面定义的 getOboToken 从 Microsoft 身份平台获取 OBO 令牌。

**初始化 Graph 客户端：** 使用 OBO 令牌，通过上面定义的 initGraphClient 初始化 Microsoft Graph 客户端。

**文档搜索：** 构建搜索查询并将其发送到 Microsoft Graph API，根据 searchTerm 查找文档。

**文档处理**：对于搜索返回的每个文档：

- 使用 getDriveItemContent 检索文档内容。

- 将文档转换为 base64 字符串，并重新组织以匹配 `openaiFileResponse` 结构。

**响应**：函数通过 HTTP 响应将它们返回。
```javascript
module.exports = async function (context, req) {
   // const query = req.query.query || (req.body && req.body.query);
   const searchTerm = req.query.searchTerm || (req.body && req.body.searchTerm);
   if (!req.headers.authorization) {
       context.res = {
           status: 400,
           body: 'Authorization header is missing'
       };
       return;
   }
   /// The below takes the token passed to the function, to use to get an OBO token.
   const bearerToken = req.headers.authorization.split(' ')[1];
   let accessToken;
   try {
       accessToken = await getOboToken(bearerToken);
   } catch (error) {
       context.res = {
           status: 500,
           body: `Failed to obtain OBO token: ${error.message}`
       };
       return;
   }
   // Initialize the Graph Client using the initGraphClient function defined above
   let client = initGraphClient(accessToken);
   // this is the search body to be used in the Microsoft Graph Search API: https://learn.microsoft.com/en-us/graph/search-concept-files
   const requestBody = {
       requests: [
           {
               entityTypes: ['driveItem'],
               query: {
                   queryString: searchTerm
               },
               from: 0,
               // the below is set to summarize the top 10 search results from the Graph API, but can configure based on your documents.
               size: 10
           }
       ]
   };


   try {
       // This is where we are doing the search
       const list = await client.api('/search/query').post(requestBody);
       const processList = async () => {
           // This will go through and for each search response, grab the contents of the file and summarize with gpt-3.5-turbo
           const results = [];
           await Promise.all(list.value[0].hitsContainers.map(async (container) => {
               for (const hit of container.hits) {
                   if (hit.resource["@odata.type"] === "#microsoft.graph.driveItem") {
                       const { name, id } = hit.resource;
                       // The below is where the file lives
                       const driveId = hit.resource.parentReference.driveId;
                       // we use the helper function we defined above to get the contents, convert to base64, and restructure it
                       const contents = await getDriveItemContent(client, driveId, id, name);
                       results.push(contents)
               }
           }));
           return results;
       };
       let results;
       if (list.value[0].hitsContainers[0].total == 0) {
           // Return no results found to the API if the Microsoft Graph API returns no results
           results = 'No results found';
       } else {
           // If the Microsoft Graph API does return results, then run processList to iterate through.
           results = await processList();
           // this is where we structure the response so ChatGPT knows they are files
           results = {'openaiFileResponse': results}
       }
       context.res = {
           status: 200,
           body: results
       };
   } catch (error) {
       context.res = {
           status: 500,
           body: `Error performing search or processing results: ${error.message}`,
       };
   }
};
```
### 定制

以下是一些可以定制的方面。

- 你可以定制 GPT 提示，让它在找不到结果时再搜索一定次数。

- 你可以通过定制搜索查询，让代码只搜索特定的 SharePoint 站点或 O365 驱动器。这有助于聚焦搜索并改善检索效果。按目前的设置，该函数会搜索已登录用户可访问的所有文件。

- 你可以更新代码，只返回特定类型的文件。例如，只返回结构化数据/CSV。

- 你可以定制在调用 Microsoft Graph 时搜索的文件数量。注意，根据[这里](https://platform.openai.com/docs/actions/getting-started)的文档，最多只能放 10 个文件。

### 注意事项

请注意，Actions 的所有限制同样适用于这里，包括返回 100K 字符或更少的内容，以及 [45 秒超时](https://platform.openai.com/docs/actions/production/timeouts)。

- 请务必阅读这里关于[返回文件](https://platform.openai.com/docs/actions/sending-files)和[文件上传](https://help.openai.com/en/articles/8555545-file-uploads-faq)的文档，因为这些限制同样适用于此。

## ChatGPT 步骤

### Custom GPT 指令

创建 Custom GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看 [Getting Started Example](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步是如何进行的。

In [ ]:
You are a Q&A helper that helps answer users questions. You have access to a documents repository through your API action. When a user asks a question, you pass in the "searchTerm" a single keyword or term you think you should use for the search.

****

Scenario 1: There are answers

If your action returns results, then you take the results from the action and try to answer the users question. 

****

Scenario 2: No results found

If the response you get from the action is "No results found", stop there and let the user know there were no results and that you are going to try a different search term, and explain why. You must always let the user know before conducting another search.

Example:

****

I found no results for "DEI". I am now going to try [insert term] because [insert explanation]

****

Then, try a different searchTerm that is similar to the one you tried before, with a single word. 

Try this three times. After the third time, then let the user know you did not find any relevant documents to answer the question, and to check SharePoint. 
Be sure to be explicit about what you are searching for at each step.

****

In either scenario, try to answer the user's question. If you cannot answer the user's question based on the knowledge you find, let the user know and ask them to go check the HR Docs in SharePoint. 

### OpenAPI Schema

创建 Custom GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看 [Getting Started Example](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步是如何进行的。

它期望的响应需与我们[这里](https://platform.openai.com/docs/actions/sending-files)文档中的文件检索结构相匹配，并传入一个 `searchTerm` 参数来指导搜索。
>请务必根据上方截图中复制的链接，替换函数应用名称、函数名称和 code

In [ ]:
openapi: 3.1.0
info:
  title: SharePoint Search API
  description: API for searching SharePoint documents.
  version: 1.0.0
servers:
  - url: https://{your_function_app_name}.azurewebsites.net/api
    description: SharePoint Search API server
paths:
  /{your_function_name}?code={enter your specific endpoint id here}:
    post:
      operationId: searchSharePoint
      summary: Searches SharePoint for documents matching a query and term.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                searchTerm:
                  type: string
                  description: A specific term to search for within the documents.
      responses:
        '200':
          description: A CSV file of query results encoded in base64.
          content:
            application/json:
              schema:
                type: object
                properties:
                  openaiFileResponseData:
                    type: array
                    items:
                      type: object
                      properties:
                        name:
                          type: string
                          description: The name of the file.
                        mime_type:
                          type: string
                          description: The MIME type of the file.
                        content:
                          type: string
                          format: byte
                          description: The base64 encoded contents of the file.
        '400':
          description: Bad request when the SQL query parameter is missing.
        '413':
          description: Payload too large if the response exceeds the size limit.
        '500':
          description: Server error when there are issues executing the query or encoding the results.

## 身份验证说明

以下是为该第三方应用设置身份验证的说明。有疑问？请查看 [Getting Started Example](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步是如何进行的。

*有关身份验证的更详细说明，请参见上文以及 [Azure Function cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function)。*

## 常见问题与故障排除

- 为什么你的代码中使用的是 Microsoft Graph API，而不是 [SharePoint API](https://learn.microsoft.com/en-us/sharepoint/dev/sp-add-ins/get-to-know-the-sharepoint-rest-service?tabs=csom)？

  - SharePoint API 属于旧版 API——根据 Microsoft [这里](https://learn.microsoft.com/en-us/sharepoint/dev/apis/sharepoint-rest-graph)的文档，"For SharePoint Online, innovation using a REST API against SharePoint is driven via the Microsoft Graph REST API's."（对于 SharePoint Online，针对 SharePoint 使用 REST API 的创新由 Microsoft Graph REST API 推动。）Graph API 为我们提供了更大的灵活性，而 SharePoint API 仍会遇到[为什么需要这样做，而不是直接与 Microsoft Graph API 交互？](#why-is-this-necessary-instead-of-interacting-with-the-microsoft-api-directly)一节中列出的同样的文件问题。

- 它支持哪些类型的文件？

  它遵循与[这里](https://help.openai.com/en/articles/8555545-file-uploads-faq)关于文件上传的文档相同的指南。

- 为什么我需要请求 OBO 令牌？

  - 当你尝试使用与 Azure Function 身份验证相同的令牌来进行 Graph API 身份验证时，会得到一个 "invalid audience" 令牌错误。这是因为该令牌的 audience 只能是 user\_impersonation。

  - 为了解决这个问题，函数会使用 [On Behalf Of 流程](https://learn.microsoft.com/en-us/entra/identity-platform/v2-oauth2-on-behalf-of-flow)，在应用内请求一个作用域为 Files.Read.All 的新令牌。它会继承已登录用户的权限，这意味着该函数只会搜索已登录用户有权访问的文件。

  - 我们有意在每次请求时都请求一个新的 On Behalf Of 令牌，因为 Azure Function Apps 被设计为无状态的。你也可以将其与 Azure Key Vault 集成，以存储密钥并以编程方式检索。

*有没有你希望我们优先支持的集成？我们的集成中有错误吗？请在我们的 github 上提交 PR 或 issue，我们会进行查看。*